In [ ]:
# Data sources:
# Work/labour force survey 2023:
# https://statbel.fgov.be/fr/themes/emploi-formation/marche-du-travail/emploi-et-chomage/plus

# Yearly teleworking survey:
# https://statbel.fgov.be/fr/themes/emploi-formation/marche-du-travail/travail-domicile#figures

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
# Telework industry lookup: mapping NACE letters to WFS counts (T, H, P, J)
# T = toujours, H = habituellement, P = parfois, J = jamais
# Work force survey 2023, national counts
telework_industry_detail = {
    "A": [1419.644757, 258.6786802, 1120.136097, 10249.35402],
    "B": [0, 68.91323859, 258.3317214, 3118.921958],
    "C": [5822.311808, 33819.25357, 90472.41435, 387916.3757],
    "D": [468.4782487, 7905.946825, 7845.142861, 13709.96015],
    "E": [95.24958074, 3033.855786, 4747.558455, 21948.21121],
    "F": [2089.176894, 6557.334821, 21184.66083, 187705.6772],
    "G": [7537.396288, 26708.34598, 56591.88254, 393695.4277],
    "H": [1791.541789, 11559.18177, 28068.69545, 203241.4586],
    "I": [1067.869443, 378.890824, 2826.59773, 129238.655],
    "J": [8851.935481, 59778.33132, 50019.63404, 38840.10999],
    "K": [4922.187024, 50404.65057, 32680.1054, 40579.87357],
    "L": [1160.417394, 3812.075277, 8289.731816, 13341.35787],
    "M": [6232.294107, 39700.77425, 72702.40435, 78722.88546],
    "N": [2932.448168, 23230.56691, 38095.7901, 188926.5396],
    "O": [3390.227551, 78349.86748, 107257.8133, 216171.4709],
    "P": [7926.886142, 46804.8274, 170574.9963, 203051.8583],
    "Q": [7483.746297, 12908.77965, 49259.45166, 511470.3798],
    "R": [583.8584028, 2721.473764, 14385.07135, 47469.36709],
    "S": [1660.928652, 7667.575593, 18689.28314, 30740.16437],
    "T": [1131.962435, 0, 97.04769968, 3077.233942],
    "U": [755.8240154, 22802.64446, 21639.95953, 15473.75698]
}

telework_total = {
    "Belgium": [67324.38448, 438471.9682, 796806.7088, 2738689.039],
    "BCR": [10035.74586, 69133.2125, 97633.21467, 243345.2156],
}

In [ ]:
# Percentage of workers who telework at least sometimes by gender
# Teleworking survey, year 2023, Brussels counts
telework_gender = {
    "F": 45.5,
    "M": 39.1,
    "Total": 42.1
}

In [ ]:
# Percentage of workers who telework at least sometimes by educational attainment
# Teleworking survey, year 2023, Belgian counts
telework_education = {
    "Primary": 4.9,
    "Secondary": 13.8,
    "Tertiary": 53.0,
    "Total": 32.2
}

In [ ]:
# Sum together for codes that we have grouped in the population
telework_industry_groups = {
    "A": telework_industry_detail["A"],
    "B-E": np.sum([np.array(telework_industry_detail[k]) for k in ["B", "C", "D", "E"]], axis=0).tolist(),
    "F": telework_industry_detail["F"],
    "G-I": np.sum([np.array(telework_industry_detail[k]) for k in ["G", "H", "I"]], axis=0).tolist(),
    "J": telework_industry_detail["J"],
    "K": telework_industry_detail["K"],
    "L": telework_industry_detail["L"],
    "M_N": np.sum([np.array(telework_industry_detail[k]) for k in ["M", "N"]], axis=0).tolist(),
    "O-Q": np.sum([np.array(telework_industry_detail[k]) for k in ["O", "P", "Q"]], axis=0).tolist(),
    "R-U": np.sum([np.array(telework_industry_detail[k]) for k in ["R", "S", "T", "U"]], axis=0).tolist(),
    "UN": telework_total["Belgium"],
}

In [ ]:
# Day-equivalent weights for each frequency category
# Toujours=5.0d/wk, Habituellement=3.0d/wk, Parfois=1.0d/wk, Jamais=0d/wk
FREQ_DAYS = [5.0, 3.0, 1.5, 0.0]

In [ ]:
def compute_p_today(counts):
    """
    Convert [Jamais, Parfois, Habituellement, Toujours] counts
    to p(telework on a given day).
    p_today = sum(n_i * days_i/5) / sum(n_i)
    Returns 0 if total is 0 (unreliable cell).
    """
    total = sum(counts)
    if total == 0:
        return None   # unreliable — will use IBSA fallback
    return sum(c * d / 5.0 for c, d in zip(counts, FREQ_DAYS)) / total

def build_telework_lookup():
    """
    Build {nace_group: p_today} from WFS counts
    """
    lookup = {}
    for nace, counts in telework_industry_groups.items():
        p = compute_p_today(counts)
        lookup[nace] = p
    return lookup

TELEWORK_P_TODAY = build_telework_lookup()

# Gender multiplier -- 42.1 total average, 45.5 for F, 39.1 for M
# Not used in the end
TELEWORK_GENDER_MULTIPLIER = {
    'M': telework_gender["M"] / telework_gender["Total"],       
    'F': telework_gender["F"] / telework_gender["Total"],
}

def get_telework_gender_mult(sex):
    return TELEWORK_GENDER_MULTIPLIER.get(sex, 1.0)

BRUSSELS_BOOST = compute_p_today(telework_total["BCR"]) / compute_p_today(telework_total["Belgium"])

EDUCATION_MULTIPLIER = {
    'Primary': telework_education["Primary"] / telework_education["Total"],
    'Secondary': telework_education["Secondary"] / telework_education["Total"],
    'Tertiary': telework_education["Tertiary"] / telework_education["Total"],
    'Total': 1.0,
}

In [ ]:
education_mapping = {
    'ISCED11_1': 'Primary',
    'ISCED11_2': 'Secondary',
    'ISCED11_3': 'Secondary',
    'ISCED11_4': 'Secondary',
    'ISCED11_5': 'Tertiary',
    'ISCED11_6': 'Tertiary',
    'ISCED11_7': 'Tertiary',
    'ISCED11_8': 'Tertiary',
    'UNK': 'Total',
    'NONE': 'Total',
}

In [ ]:
def assign_telework(pop, random_seed=42):
    """
    Assign p(telework today) and draw binary telework_today flag.

    Parameters
    ----------
    pop : pd.DataFrame
        Must have columns: industry, sex, education

    Returns
    -------
    pop : pd.DataFrame with new columns:
        p_telework_today   — probability of teleworking today
        telework_today     — binary (1 = teleworking, skip commute)
    """
    rng = np.random.default_rng(random_seed)
    pop = pop.copy()

    industry  = pop['industry'].values
    education = pop['education'].values

    p_today = np.zeros(len(pop))

    for i in range(len(pop)):
        nace = industry[i] if industry[i] in TELEWORK_P_TODAY else 'UN'
        education_group = education_mapping.get(education[i], 'Total')
        base = TELEWORK_P_TODAY[nace] * EDUCATION_MULTIPLIER[education_group] * (BRUSSELS_BOOST+1)/2

        p_today[i] = min(base, 0.95)

    pop['p_telework_today'] = p_today
    pop['telework_today']   = rng.binomial(1, p_today)

    return pop

In [ ]:
# Assign telework and save results
pop = pd.read_csv('../5_company_cars/output/workers_with_company_cars.csv')
pop = assign_telework(pop)

# Save the full population with telework assignment
pop.to_csv('output/workers_with_telework.csv', index=False)

# Commuting population (non-teleworkers)
commuters = pop[pop['telework_today'] == 0].copy()
commuters.to_csv('output/workers_not_teleworking.csv', index=False)